#  Heatwave Early-Warning System with Prefect
### Data-pipeline orchestration + ML-pipeline orchestration in one notebook

**Application:** pull 3 years of daily weather for 6 Indian cities, clean it, build features, train & compare models,
pick the best one, and raise **HEATWAVE alerts** (forecast max ≥ 40 °C).

```
heatwave-early-warning  (parent flow)
├── data-pipeline-etl   (subflow)  extract ×6 in parallel → validate/clean → features → load CSV
└── ml-pipeline-training (subflow) split → train ×2 in parallel → evaluate → select & save → alerts
```

**Prefect ideas you will see:** `@task`, `@flow`, subflows, parallel `.submit()`, retries, caching, logging, artifacts.

Data source: free [Open-Meteo archive API](https://open-meteo.com) (no key). If there is no internet, the pipeline falls back to simulated data automatically.

In [ ]:
!pip install -q prefect scikit-learn tabulate

In [ ]:
import os, subprocess, time, requests

# True  -> runs show up in your Prefect Cloud dashboard (app.prefect.cloud)
# False -> local Prefect server inside Colab (no account needed, but no web UI you can open)
USE_PREFECT_CLOUD = True

if USE_PREFECT_CLOUD:
    os.environ["PREFECT_API_URL"] =
    os.environ["PREFECT_API_KEY"] =
else:
    os.environ["PREFECT_API_URL"] = "http://127.0.0.1:4200/api"
    server = subprocess.Popen(
        ["prefect", "server", "start", "--host", "127.0.0.1", "--port", "4200"],
        stdout=open("prefect_server.log", "w"), stderr=subprocess.STDOUT,
    )
    for i in range(90):                            # first start can take 30-60 s
        try:
            if requests.get("http://127.0.0.1:4200/api/health", timeout=2).ok:
                print(f"Local Prefect server ready after {i*2} s")
                break
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(2)

# Check the connection (prefect must be imported AFTER the env vars are set)
import prefect
from prefect.client.orchestration import get_client
with get_client(sync_client=True) as client:
    client.read_flows(limit=1)
print("Connected to:", os.environ["PREFECT_API_URL"].split("/workspaces")[0][:45], "| Prefect", prefect.__version__)

Connected to: https://api.prefect.cloud/api/accounts/1db076 | Prefect 3.8.8


### 1. Imports and settings
With `USE_PREFECT_CLOUD = True`, every flow run, task log and artifact appears live at **app.prefect.cloud → Runs**. With `False`, they are stored in a local server inside Colab.

In [ ]:
import random
import pandas as pd
import numpy as np
import requests
from datetime import date, timedelta

from prefect import flow, task, get_run_logger
from prefect.artifacts import create_markdown_artifact, create_table_artifact
from prefect.cache_policies import INPUTS

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import joblib

CITIES = {
    "Delhi":     (28.61, 77.21),
    "Mumbai":    (19.08, 72.88),
    "Chennai":   (13.08, 80.27),
    "Kolkata":   (22.57, 88.36),
    "Bengaluru": (12.97, 77.59),
    "Nagpur":    (21.15, 79.09),
}
START = "2023-01-01"
END = "2025-12-31"
HEATWAVE_LIMIT = 40.0   # degrees C

### 2. Data pipeline (ETL)
* `extract_city` has **retries=3** (network calls fail) and **cache_policy=INPUTS** (same city + dates → reuse result, no re-download).
* `.submit()` runs all 6 downloads **in parallel**.
* `create_table_artifact` saves a data-quality report inside Prefect.

In [ ]:
#Real weather data from Open-Meteo
@task(retries=3, retry_delay_seconds=2, cache_policy=INPUTS)  # Retry failures up to 3 times, wait 2 seconds, and cache by task inputs.
def extract_city(city, lat, lon):  # Extract daily weather for the given city and coordinates.
    """Download daily weather for one city from the free Open-Meteo archive API."""  # Describe the function's purpose.
    logger = get_run_logger()  # Get the logger for the current Prefect run.
    url = "https://archive-api.open-meteo.com/v1/archive"  # Set the historical weather API endpoint.
    params = {  # Build the parameters sent to the weather API.
        "latitude": lat, "longitude": lon,  # Specify the city's geographic coordinates.
        "start_date": START, "end_date": END,  # Specify the requested date range.
        "daily": "temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum",  # Request daily temperature, humidity, and rainfall.
        "timezone": "Asia/Kolkata",  # Use Indian local time for daily weather records.
    }  # Finish the API parameter dictionary.
    try:  # Attempt to download and process real weather data.
        r = requests.get(url, params=params, timeout=30)  # Send the request with a 30-second timeout.
        r.raise_for_status()  # Raise an error if the server returns an unsuccessful HTTP status.
        df = pd.DataFrame(r.json()["daily"])  # Extract daily JSON data and convert it into a Pandas table.
        logger.info(f"{city}: downloaded {len(df)} days from Open-Meteo")  # Log the city and number of downloaded daily rows.
    except Exception as e:  # Handle errors raised during the download or response processing.
        logger.warning(f"{city}: API failed ({type(e).__name__}) -> using simulated data")  # Log the error type and announce the fallback.
        df = simulate_city(city, lat)  # Generate simulated weather data when the API attempt fails.
    df["city"] = city  # Add the city name to every row.
    return df  # Return the real or simulated weather table.

# Request times out, internet fails, or response processing fails : Simulated weather data from simulate_city()
def simulate_city(city, lat):  # Generate simulated daily weather for a city using its latitude.
    """Seasonal fake weather: hotter in May, hotter for northern cities."""  # Describe the simulated seasonal pattern.
    rng = np.random.default_rng(abs(hash(city)) % 1000)  # Create a random generator seeded from the city name within this Python process.
    days = pd.date_range(START, END, freq="D")  # Create one date for each day in the requested range.
    doy = days.dayofyear.values  # Convert dates into day-of-year numbers, such as January 1 → 1.
    season = np.sin(2 * np.pi * (doy - 60) / 365)  # Create a seasonal wave that peaks around late May.
    tmax = 31 + 3 * season + (lat - 12) * 0.5 * (season + 0.3) + rng.normal(0, 1.5, len(days))  # Combine baseline temperature, season, latitude, and random variation.
    df = pd.DataFrame({  # Build a table containing the simulated daily weather.
        "time": days.strftime("%Y-%m-%d"),  # Format each date as a year-month-day string.
        "temperature_2m_max": tmax.round(1),  # Store maximum temperatures rounded to one decimal place.
        "temperature_2m_min": (tmax - 9 + rng.normal(0, 1, len(days))).round(1),  # Generate minimum temperatures approximately 9°C below the maximum.
        "relative_humidity_2m_mean": np.clip(60 - 15 * season + rng.normal(0, 8, len(days)), 10, 100).round(0),  # Generate humidity and restrict it to 10–100%.
        "precipitation_sum": np.clip(rng.gamma(0.4, 6, len(days)) * (season < 0), 0, None).round(1),  # Generate nonnegative rainfall only when the seasonal wave is negative.
    })  # Finish creating the weather table.
    bad = rng.choice(len(df), 15, replace=False)  # Select 15 distinct rows to introduce deliberate data-quality problems.
    df.loc[bad[:10], "temperature_2m_max"] = np.nan  # Replace maximum temperatures in the first 10 selected rows with missing values.
    df.loc[bad[10:], "temperature_2m_max"] = 99.9  # Set the remaining 5 selected maximum temperatures to unrealistic values.
    return df  # Return the simulated weather table containing deliberate dirty data.


@task  # Register this function as a Prefect task.
def validate_and_clean(df):  # Check and clean the city's weather table.
    """Data quality checks: missing values and impossible temperatures."""  # Describe the cleaning checks.
    logger = get_run_logger()  # Get the logger for the current Prefect run.
    city = df["city"].iloc[0]  # Read the city name from the first row.
    missing = int(df["temperature_2m_max"].isna().sum())  # Count missing maximum temperatures before cleaning.
    impossible = int((df["temperature_2m_max"] > 55).sum())  # Count maximum temperatures above the chosen 55°C limit.

    df = df.copy()  # Create a copy to avoid changing the original input table.
    df.loc[df["temperature_2m_max"] > 55, "temperature_2m_max"] = np.nan  # Replace temperatures above 55°C with missing values.
    df["temperature_2m_max"] = df["temperature_2m_max"].interpolate().bfill().ffill()  # Estimate gaps between values, then fill any remaining edge gaps.
    df = df.fillna(df.mean(numeric_only=True))  # Fill remaining missing numeric values with their column averages.

    logger.info(f"{city}: fixed {missing} missing and {impossible} impossible values")  # Log the original counts of missing and outlier temperatures.
    return df, {"city": city, "rows": len(df), "missing_fixed": missing, "outliers_fixed": impossible}  # Return the cleaned table and a quality summary.


@task  # Register this function as a Prefect task.
def make_features(df):  # Create input features and tomorrow's temperature target.
    """Target = tomorrow's max temperature. Features = today and recent days."""  # Describe what the model will learn to predict.
    df = df.sort_values("time").copy()  # Arrange rows chronologically and copy the table.
    df["date"] = pd.to_datetime(df["time"])  # Convert date strings into Pandas datetime values.
    df["month"] = df["date"].dt.month  # Extract the month number, from 1 to 12, as a seasonal feature.
    df["tmax_lag1"] = df["temperature_2m_max"].shift(1)  # Add yesterday's maximum temperature to today's row.
    df["tmax_avg7"] = df["temperature_2m_max"].rolling(7).mean()  # Average today's maximum temperature and the previous six rows.
    df["target_tomorrow"] = df["temperature_2m_max"].shift(-1)  # Add tomorrow's maximum temperature as the value to predict.
    return df.dropna()  # Remove rows with missing values, including initial rolling-window rows and the final target row.

@task  # Register this function as a Prefect task.
def load_to_warehouse(frames):  # Combine the weather tables supplied in the frames list.
    """'Load' step: combine all cities into one table and save as CSV."""  # Describe the pipeline's loading step.
    data = pd.concat(frames, ignore_index=True)  # Stack all city tables vertically and create a fresh row index.
    data.to_csv("weather_warehouse.csv", index=False)  # Save the combined table as CSV without the row index.
    get_run_logger().info(f"Warehouse saved: {len(data)} rows, {data['city'].nunique()} cities")  # Log the total rows and number of distinct cities.
    return data  # Return the combined table for downstream processing.


@flow(name="data-pipeline-etl", log_prints=True)  # Register the complete Prefect flow and capture print statements as logs.
def data_pipeline():  # Download, clean, transform, and save weather data for all cities.
    futures = [extract_city.submit(c, lat, lon) for c, (lat, lon) in CITIES.items()]  # Submit each city's extraction task for concurrent execution.
    raw = [f.result() for f in futures]  # Wait for each extraction task and collect its weather table.

    cleaned, reports = [], []  # Create lists for processed weather tables and data-quality summaries.
    for df in raw:  # Process each city's downloaded or simulated weather table.
        clean_df, report = validate_and_clean(df)  # Clean missing/outlier values and obtain a quality summary.
        cleaned.append(make_features(clean_df))  # Create prediction features and add the resulting table to the list.
        reports.append(report)  # Add this city's quality summary to the reports list.

    create_table_artifact(  # Publish the city-level cleaning summaries in Prefect.
        key="data-quality-report",  # Name the artifact so its versions can be tracked.
        table=reports,  # Supply one quality-report row per city.
        description="Rows cleaned per city"  # Describe the contents of the report.
    )  # Finish publishing the report.
    return load_to_warehouse(cleaned)  # Combine processed city tables, save the CSV, and return the combined data.

### 3. ML pipeline
* **Time-based split:** train 2023–24, test 2025 (never shuffle time series!).
* Two models trained in parallel; the lowest MAE wins and is saved to `best_model.joblib`.
* A markdown artifact stores the leaderboard + alerts.

In [ ]:
FEATURES = ["temperature_2m_max", "temperature_2m_min", "relative_humidity_2m_mean",
            "precipitation_sum", "month", "tmax_lag1", "tmax_avg7"]


@task
def split_data(data):
    """Time-based split: train on 2023-2024, test on 2025 (no peeking into the future)."""
    train = data[data["date"] < "2025-01-01"]
    test = data[data["date"] >= "2025-01-01"]
    print(f"Train: {len(train)} rows | Test: {len(test)} rows")
    return train, test


@task
def train_model(name, train):
    if name == "linear_regression":
        model = LinearRegression()
    else:
        model = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
    model.fit(train[FEATURES], train["target_tomorrow"])
    return name, model


@task
def evaluate_model(name, model, test):
    pred = model.predict(test[FEATURES])
    mae = mean_absolute_error(test["target_tomorrow"], pred)
    print(f"{name}: MAE = {mae:.2f} degrees C")
    return {"model": name, "mae": round(mae, 3)}


@task  # Register this function as a Prefect task.
def select_and_register(results, models):  # Select the lowest-error model and save it.
    """Pick the model with the lowest error and save it (a tiny 'model registry')."""  # Describe model selection and storage.
    print(results)
    print(models)
    best = min(results, key=lambda r: r["mae"])  # Find the result with the lowest mean absolute error (MAE).
    joblib.dump(models[best["model"]], "best_model.joblib")  # Retrieve the winning model by name and save it to a file.
    print(f"Best model: {best['model']} (MAE {best['mae']})")  # Display the winning model's name and error.
    return best  # Return the winning result dictionary.


@task  # Register this function as a Prefect task.
def heatwave_alerts(data, model):  # Predict tomorrow's temperature for each city and flag high forecasts.
    """Use the latest day of each city to forecast tomorrow and raise alerts."""  # Describe forecasting and alert generation.
    latest = data.sort_values("date").groupby("city").tail(1).copy()  # Select the latest available row for each city.
    latest["forecast_tmax"] = model.predict(latest[FEATURES]).round(1)  # Predict tomorrow's maximum temperature and round to one decimal place.
    latest["date"] = latest["date"].dt.date  # Convert each input timestamp into a date without a time component.
    latest["alert"] = np.where(latest["forecast_tmax"] >= HEATWAVE_LIMIT, "HEATWAVE", "normal")  # Flag forecasts at or above the configured temperature threshold.
    return latest[["city", "date", "temperature_2m_max", "forecast_tmax", "alert"]]  # Return the city, input date, observed temperature, forecast, and alert flag.

@flow(name="ml-pipeline-training", log_prints=True)  # Register the ML workflow and capture printed output in Prefect logs.
def ml_pipeline(data):  # Train models, compare their errors, save the winner, and generate forecast alerts.
    train, test = split_data(data)  # Divide the supplied data into training and evaluation partitions.

    futures = [train_model.submit(n, train) for n in ["linear_regression", "random_forest"]]  # Submit both models for concurrent training.
    models = dict(f.result() for f in futures)  # Wait for training and build a dictionary of model names and fitted models.

    results = [evaluate_model(n, m, test) for n, m in models.items()]  # Evaluate each fitted model on the same held-out data.
    best = select_and_register(results, models)  # Select the lowest-MAE model and save it to disk.

    may_day = data[data["date"] == "2025-05-15"]  # Select each city's available record for 15 May 2025.
    alerts = heatwave_alerts(may_day, models[best["model"]])  # Use the winning model to predict the next day's temperature and flag heatwave risk.
    print(alerts.to_string(index=False))  # Display the forecast and alert table without row indices.

    create_markdown_artifact(  # Publish the model comparison and forecasts as a readable Prefect report.
        key="ml-report",  # Assign an artifact key for tracking report versions.
        markdown=f"## Model leaderboard\n\n{pd.DataFrame(results).to_markdown(index=False)}\n\n"  # Format all model evaluation results as a Markdown table.
                 f"**Winner:** {best['model']}\n\n## Heatwave forecast (from 15-May-2025)\n\n"  # Add the winning model and forecast context.
                 f"{alerts.to_markdown(index=False)}",  # Append the city-level forecast and alert table.
    )  # Finish publishing the report.
    return best, alerts  # Return the winning model's result and the generated alerts.

### 4. Parent flow — orchestrates both pipelines
Each subflow gets its own run, state and logs.

In [ ]:
@flow(name="heatwave-early-warning")
def heatwave_system():
    data = data_pipeline()          # subflow 1: ETL
    best, alerts = ml_pipeline(data)  # subflow 2: ML
    return alerts

### 5. Run it

In [ ]:
alerts = heatwave_system()
alerts

INFO:prefect.flow_runs:Beginning flow run 'whimsical-sunfish' for flow 'heatwave-early-warning'
INFO:prefect.flow_runs:View at https://app.prefect.cloud/account/1db07649-353f-4e19-8021-644bee1b8296/workspace/d741b578-a867-4c10-b2b4-fb97fd4f7e80/runs/flow-run/06acf48b-5886-74df-8000-e601c768127a
INFO:prefect.flow_runs:Beginning subflow run 'delectable-vole' for flow 'data-pipeline-etl'
INFO:prefect.flow_runs:View at https://app.prefect.cloud/account/1db07649-353f-4e19-8021-644bee1b8296/workspace/d741b578-a867-4c10-b2b4-fb97fd4f7e80/runs/flow-run/06acf48b-6a28-7b43-8000-1149c4706a2f
INFO:prefect.task_runs:Finished in state Cached(type=COMPLETED)
INFO:prefect.task_runs:Finished in state Cached(type=COMPLETED)
INFO:prefect.task_runs:Finished in state Cached(type=COMPLETED)
INFO:prefect.task_runs:Finished in state Cached(type=COMPLETED)
INFO:prefect.task_runs:Finished in state Cached(type=COMPLETED)
INFO:prefect.task_runs:Finished in state Cached(type=COMPLETED)
INFO:prefect.task_runs:Delhi

,city,date,temperature_2m_max,forecast_tmax,alert
859,Delhi,2025-05-15,41.9,41.6,HEATWAVE
1948,Mumbai,2025-05-15,33.2,33.3,normal
3037,Chennai,2025-05-15,36.5,36.5,normal
4126,Kolkata,2025-05-15,38.2,38.0,normal
5215,Bengaluru,2025-05-15,33.3,32.9,normal
6304,Nagpur,2025-05-15,37.3,37.6,normal


### 6. Caching demo — run again
Watch the logs: `extract_city` now shows **`Cached`** and finishes much faster, because inputs didn't change.

In [ ]:
import time
t = time.time()
heatwave_system()
print(f'Second run took {time.time()-t:.1f} s')

INFO:prefect.flow_runs:Beginning flow run 'strong-kiwi' for flow 'heatwave-early-warning'
INFO:prefect.flow_runs:View at https://app.prefect.cloud/account/1db07649-353f-4e19-8021-644bee1b8296/workspace/d741b578-a867-4c10-b2b4-fb97fd4f7e80/runs/flow-run/06acf37c-830e-7383-8000-7be4e3296bd4
INFO:prefect.flow_runs:Beginning subflow run 'spirited-eagle' for flow 'data-pipeline-etl'
INFO:prefect.flow_runs:View at https://app.prefect.cloud/account/1db07649-353f-4e19-8021-644bee1b8296/workspace/d741b578-a867-4c10-b2b4-fb97fd4f7e80/runs/flow-run/06acf37c-951b-77de-8000-45f0bdf63d51
INFO:prefect.task_runs:Finished in state Cached(type=COMPLETED)
INFO:prefect.task_runs:Finished in state Cached(type=COMPLETED)
INFO:prefect.task_runs:Finished in state Cached(type=COMPLETED)
INFO:prefect.task_runs:Finished in state Cached(type=COMPLETED)
INFO:prefect.task_runs:Finished in state Cached(type=COMPLETED)
INFO:prefect.task_runs:Finished in state Cached(type=COMPLETED)
INFO:prefect.task_runs:Delhi: fixed

Second run took 6.8 s


### 7. Retries demo — a flaky sensor API
The task fails twice, Prefect retries automatically, and the flow still succeeds.

In [ ]:
attempts = {"n": 0}  # Store the number of attempts made in this Python process.

@task(retries=4, retry_delay_seconds=1)  # Allow up to 4 retries after the initial attempt, waiting 1 second between retries.
def flaky_sensor_api():  # Simulate a sensor API that fails twice before succeeding.
    attempts["n"] += 1  # Increase the attempt counter each time the task executes.
    print(f"Attempt {attempts['n']} ...")  # Display the current attempt number.
    if attempts["n"] < 3:  # Fail on attempts 1 and 2.
        raise ConnectionError("Sensor gateway timed out!")  # Raise an exception so Prefect retries the task.
    return "data received"  # Succeed on attempt 3 and return the simulated response.

@flow(log_prints=True)  # Register the workflow and capture printed messages in Prefect logs.
def retry_demo():  # Define a workflow demonstrating automatic task retries.
    return flaky_sensor_api()  # Run the task and return its result once it succeeds.

retry_demo()  # Start the workflow; the task succeeds on its third attempt.

INFO:prefect.flow_runs:Beginning flow run 'primitive-hippo' for flow 'retry-demo'
INFO:prefect.flow_runs:View at https://app.prefect.cloud/account/1db07649-353f-4e19-8021-644bee1b8296/workspace/d741b578-a867-4c10-b2b4-fb97fd4f7e80/runs/flow-run/06acf37d-8f88-75f2-8000-138fc05bcba5
INFO:prefect.task_runs:Attempt 1 ...
INFO:prefect.task_runs:Task run failed with exception: ConnectionError('Sensor gateway timed out!') - Retry 1/4 will start 1 second(s) from now
INFO:prefect.task_runs:Attempt 2 ...
INFO:prefect.task_runs:Task run failed with exception: ConnectionError('Sensor gateway timed out!') - Retry 2/4 will start 1 second(s) from now
INFO:prefect.task_runs:Attempt 3 ...
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.flow_runs:Finished in state Completed()


'data received'

### 8. See what Prefect recorded
Artifacts (data-quality table and ML report) and the history of flow runs.

In [ ]:
!prefect artifact ls
!prefect artifact inspect ml-report

                                   Artifacts                                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃                                 ID ┃ Key               ┃ Type   ┃ Updated    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ 55007b46-9935-45e1-ad57-88e6c88ad… │ data-quality-rep… │ table  │ 2 minutes… │
│ 94ce88b8-55d3-418e-826c-a38c0cf6f… │ ml-report         │ markd… │ 2 minutes… │
│ 725bb1a5-0359-426c-b0e7-495676028… │ mnist-best-model  │ markd… │ 5 hours a… │
│ 3c055b08-2d06-4e9a-bd23-0f64fc91b… │ mnist-epoch-metr… │ table  │ 5 hours a… │
│ b000dafa-717b-4367-9fd8-b3012bc6c… │ mnist-run-compar… │ table  │ 5 hours a… │
│ 7a5cb5e2-ef18-434a-9ada-178a910da… │ mnist-trial-metr… │ table  │ 5 hours a… │
└────────────────────────────────────┴───────────────────┴────────┴────────────┘
                   List Artifacts using `prefect artifact ls`                   
[
    {
        'id': '94ce8

In [ ]:
!prefect flow-run ls --limit 10

                                   Flow Runs                                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━┳━━━━━━━━━┓
┃                            ID ┃ Flow           ┃ Name          ┃ … ┃ When    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━╇━━━━━━━━━┩
│ 06acf37d-8f88-75f2-8000-138f… │ retry-demo     │ primitive-hi… │ … │ a minu… │
│ 06acf37c-b669-7fd1-8000-9165… │ ml-pipeline-t… │ smooth-bird   │ … │ 2 minu… │
│ 06acf37c-951b-77de-8000-45f0… │ data-pipeline… │ spirited-eag… │ … │ 2 minu… │
│ 06acf37c-830e-7383-8000-7be4… │ heatwave-earl… │ strong-kiwi   │ … │ 2 minu… │
│ 06acf37c-0b80-73e4-8000-715f… │ ml-pipeline-t… │ merciful-fir… │ … │ 2 minu… │
│ 06acf37b-e4d6-70c9-8000-536e… │ data-pipeline… │ brainy-deer   │ … │ 2 minu… │
│ 06acf37b-d1cc-73d2-8000-d305… │ heatwave-earl… │ meticulous-m… │ … │ 2 minu… │
│ 06acf2d9-832f-7de1-8000-92b6… │ ml-pipeline-t… │ ingenious-ki… │ … │ 45 min… │
│ 06acf2d9-4ae4-78aa-8000-42

### 9. Look at the outputs

In [ ]:
import pandas as pd
wh = pd.read_csv('weather_warehouse.csv')
print(wh.shape)
wh.groupby('city')['temperature_2m_max'].agg(['mean','max']).round(1)

In [ ]:
import matplotlib.pyplot as plt
for city in ["Delhi", "Mumbai", "Bengaluru"]:
    d = wh[wh.city == city]
    plt.plot(pd.to_datetime(d.time), d.temperature_2m_max, label=city, linewidth=0.8)
plt.axhline(40, color="red", linestyle="--", label="Heatwave 40°C")
plt.title("Daily max temperature"); plt.legend(); plt.show()

### 10. Exercises for students
1. Add a city (e.g. Jaipur `26.91, 75.79`) — only the `CITIES` dict changes.
2. Add a third model (`GradientBoostingRegressor`) to the training list.
3. Add a **quality gate**: if best MAE > 2.0, `raise ValueError` so the flow is marked **Failed**.
4. Change the `may_day` date to your birthday in 2025 and see the alerts.
5. Outside Colab: `heatwave_system.serve(name="daily", cron="0 6 * * *")` turns this into a scheduled daily job, and `prefect server start` gives you the web UI.